# Spaceship Titanic — Step 1: Problem Understanding + EDA

**Target:** `Transported` (bool) — binary classification

**Metric:** Accuracy on held-out test set

**Goal of this notebook:** understand the data structure, check class balance, missingness, and test our initial hypotheses before engineering any features:
- Is `CryoSleep` strongly associated with `Transported`?
- Are the spend columns just a proxy for *not* being in cryo (i.e., redundant with `CryoSleep`)?
- Does `HomePlanet` look like a class/status proxy?
- Does cabin deck letter correlate with `Transported`?
- Does group size (from `PassengerId`) carry signal independent of cabin location?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
train.head()

train shape: (8693, 14)
test shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Class balance and missingness

Before anything else: is the target balanced? An imbalanced target changes how we read accuracy (a 90/10 split makes 90% accuracy trivial and meaningless). Also check missing values per column — this tells us how much imputation work we're in for, and whether missingness itself might be informative (e.g., does `CryoSleep` missingness correlate with the target?).

In [2]:
print("Target balance:")
print(train["Transported"].value_counts(normalize=True))

print("\nMissing values (train):")
missing = train.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

print("\nDtypes:")
print(train.dtypes)

Target balance:
Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64

Missing values (train):
CryoSleep       217
ShoppingMall    208
VIP             203
HomePlanet      201
Name            200
Cabin           199
VRDeck          188
Spa             183
FoodCourt       183
Destination     182
RoomService     181
Age             179
dtype: int64

Dtypes:
PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object


## Is missingness correlated across columns?

If missing values were truly independent per-column noise, knowing that one field is missing on a row should tell you nothing about whether another field is also missing. We can test that directly: build a boolean "is-missing" indicator for each column, then look at the correlation between those indicators. A strong positive correlation between e.g. `CryoSleep`-missing and `VIP`-missing would suggest whole records are degraded together (e.g. a corrupted upload batch), not random field-level dropout — which would point us toward "drop incomplete rows" being safer than naive per-column imputation, or toward investigating *why* those rows are bad.

In [3]:
missing_cols = missing[missing > 0].index.tolist()
missing_indicator = train[missing_cols].isna()

corr = missing_indicator.corr()
print("Correlation between 'is-missing' indicators:")
print(corr.round(2))

print("\nHow many missing fields does each row have?")
rows_missing_count = missing_indicator.sum(axis=1)
print(rows_missing_count.value_counts().sort_index())

print("\nRows with >=2 missing fields:", (rows_missing_count >= 2).sum(),
      f"({(rows_missing_count >= 2).mean():.1%} of train)")

Correlation between 'is-missing' indicators:
              CryoSleep  ShoppingMall   VIP  HomePlanet  Name  Cabin  VRDeck  \
CryoSleep          1.00          0.01  0.01       -0.01  0.02   0.01    0.01   
ShoppingMall       0.01          1.00 -0.00        0.01 -0.00  -0.01    0.01   
VIP                0.01         -0.00  1.00       -0.01 -0.02   0.01   -0.02   
HomePlanet        -0.01          0.01 -0.01        1.00  0.01   0.01   -0.02   
Name               0.02         -0.00 -0.02        0.01  1.00  -0.01   -0.00   
Cabin              0.01         -0.01  0.01        0.01 -0.01   1.00   -0.00   
VRDeck             0.01          0.01 -0.02       -0.02 -0.00  -0.00    1.00   
Spa                0.00         -0.01 -0.01       -0.01  0.01  -0.01   -0.01   
FoodCourt         -0.01         -0.01 -0.00       -0.01  0.00   0.00    0.01   
Destination       -0.00          0.00 -0.00       -0.00 -0.00  -0.00   -0.01   
RoomService       -0.01         -0.00 -0.00        0.01 -0.02  -0.00   -0.0

## Dtype audit + per-column nature

`dtype: object` for `CryoSleep` and `VIP` is misleading — printed `object` because the column mixes Python `True`/`False` with `NaN`, not because it's free text. Let's look at actual unique values per column to classify each one correctly before deciding how to handle it (boolean vs. categorical vs. continuous vs. structured-string-to-parse).

In [4]:
for col in train.columns:
    n_unique = train[col].nunique(dropna=True)
    sample_vals = train[col].dropna().unique()[:5]
    print(f"{col:14s} dtype={str(train[col].dtype):8s} n_unique={n_unique:6d}  sample={sample_vals}")

PassengerId    dtype=str      n_unique=  8693  sample=<StringArray>
['0001_01', '0002_01', '0003_01', '0003_02', '0004_01']
Length: 5, dtype: str
HomePlanet     dtype=str      n_unique=     3  sample=<StringArray>
['Europa', 'Earth', 'Mars']
Length: 3, dtype: str
CryoSleep      dtype=object   n_unique=     2  sample=[False True]
Cabin          dtype=str      n_unique=  6560  sample=<StringArray>
['B/0/P', 'F/0/S', 'A/0/S', 'F/1/S', 'F/0/P']
Length: 5, dtype: str
Destination    dtype=str      n_unique=     3  sample=<StringArray>
['TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e']
Length: 3, dtype: str
Age            dtype=float64  n_unique=    80  sample=[39. 24. 58. 33. 16.]
VIP            dtype=object   n_unique=     2  sample=[False True]
RoomService    dtype=float64  n_unique=  1273  sample=[  0. 109.  43. 303.  42.]
FoodCourt      dtype=float64  n_unique=  1507  sample=[   0.    9. 3576. 1283.   70.]
ShoppingMall   dtype=float64  n_unique=  1115  sample=[  0.  25. 371. 151.   3.]
Spa 

## Do group-mates share values? (informs smarter imputation)

`PassengerId` is formatted `gggg_pp` — group number and passenger number within group. If people traveling together tend to share `HomePlanet`, `Destination`, or even `Cabin`, then for a row with a missing value we could backfill from a groupmate *before* falling back to a global imputation (mode/median). That's strictly more informed than imputing blind. Let's measure, for groups with size > 1, what fraction of the time groupmates agree on each categorical field.

In [5]:
tmp = train.copy()
tmp["Group"] = tmp["PassengerId"].str.split("_").str[0]
group_sizes = tmp.groupby("Group")["PassengerId"].transform("size")
tmp["GroupSize"] = group_sizes

multi = tmp[tmp["GroupSize"] > 1]
print(f"Rows in a group of size > 1: {len(multi)} / {len(tmp)} ({len(multi)/len(tmp):.1%})")

for col in ["HomePlanet", "Destination", "Cabin"]:
    nunique_per_group = multi.dropna(subset=[col]).groupby("Group")[col].nunique()
    pct_consistent = (nunique_per_group == 1).mean()
    print(f"{col:12s}: among groups with non-null values, {pct_consistent:.1%} are fully consistent within the group")

Rows in a group of size > 1: 3888 / 8693 (44.7%)
HomePlanet  : among groups with non-null values, 100.0% are fully consistent within the group
Destination : among groups with non-null values, 49.2% are fully consistent within the group
Cabin       : among groups with non-null values, 70.2% are fully consistent within the group


## Does CryoSleep explain the spend columns?

Earlier hypothesis: if you're in cryo, you can't spend money, so spend should be ~0. If that holds tightly, missing spend values for cryo passengers should be imputed as 0 (not median), since 0 isn't "unknown," it's "structurally implied."

In [6]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
total_spend = train[spend_cols].sum(axis=1)

cryo_known = train.dropna(subset=["CryoSleep"]).copy()
cryo_known["TotalSpend"] = total_spend.loc[cryo_known.index]

print(cryo_known.groupby("CryoSleep")["TotalSpend"].describe())
print("\nPct of CryoSleep==True rows with TotalSpend > 0:",
      (cryo_known.loc[cryo_known["CryoSleep"] == True, "TotalSpend"] > 0).mean())

            count         mean          std  min    25%     50%     75%  \
CryoSleep                                                                 
False      5439.0  2248.299687  3245.061489  0.0  746.0  1019.0  2416.0   
True       3037.0     0.000000     0.000000  0.0    0.0     0.0     0.0   

               max  
CryoSleep           
False      35987.0  
True           0.0  

Pct of CryoSleep==True rows with TotalSpend > 0: 0.0


## Does VIP status leak through spend?

Same logic as `CryoSleep`, but weaker prior: VIP passengers might just spend more on average rather than following a hard rule. Important confound: `CryoSleep == True` forces `TotalSpend == 0` regardless of VIP status, which would dilute/hide any VIP effect if we don't control for it. So we condition on `CryoSleep == False` (awake passengers only) before comparing VIP vs non-VIP spend — otherwise a pile of sleeping non-VIPs and sleeping VIPs at 0 spend would wash out any real signal.

In [7]:
vip_known = train.dropna(subset=["VIP", "CryoSleep"]).copy()
vip_known["TotalSpend"] = total_spend.loc[vip_known.index]
awake = vip_known[vip_known["CryoSleep"] == False]

print("Awake passengers only (CryoSleep == False):")
print(awake.groupby("VIP")["TotalSpend"].describe())

print("\nPct of awake VIP==True with TotalSpend > 0:",
      (awake.loc[awake["VIP"] == True, "TotalSpend"] > 0).mean())
print("Pct of awake VIP==False with TotalSpend > 0:",
      (awake.loc[awake["VIP"] == False, "TotalSpend"] > 0).mean())

for col in spend_cols:
    print(f"\n{col} by VIP (awake only):")
    print(awake.groupby("VIP")[col].median())

Awake passengers only (CryoSleep == False):
        count         mean          std  min     25%     50%     75%      max
VIP                                                                          
False  5143.0  2160.761812  3120.593306  0.0   737.0   991.0  2346.0  35987.0
True    175.0  4866.485714  5155.348654  0.0  1647.5  3015.0  6608.0  31076.0

Pct of awake VIP==True with TotalSpend > 0: 0.96
Pct of awake VIP==False with TotalSpend > 0: 0.9033637954501264

RoomService by VIP (awake only):
VIP
False    2.5
True     2.0
Name: RoomService, dtype: float64

FoodCourt by VIP (awake only):
VIP
False      4.0
True     450.0
Name: FoodCourt, dtype: float64

ShoppingMall by VIP (awake only):
VIP
False    2.0
True     1.0
Name: ShoppingMall, dtype: float64

Spa by VIP (awake only):
VIP
False     7.0
True     94.0
Name: Spa, dtype: float64

VRDeck by VIP (awake only):
VIP
False     3.0
True     70.5
Name: VRDeck, dtype: float64


## Target relationships: CryoSleep, HomePlanet, Cabin deck/side

Now the question we opened with. For each categorical feature, look at `Transported` rate per category vs. the overall base rate (50.4%). A feature is only worth engineering hard if its categories pull meaningfully away from that base rate — small wiggles (a few points) on a near-balanced target are likely noise unless backed by a lot of rows.

In [8]:
base_rate = train["Transported"].mean()
print(f"Overall Transported rate: {base_rate:.3f}\n")

print("By CryoSleep:")
print(train.groupby("CryoSleep")["Transported"].agg(["mean", "count"]))

print("\nBy HomePlanet:")
print(train.groupby("HomePlanet")["Transported"].agg(["mean", "count"]))

cabin_parts = train["Cabin"].str.split("/", expand=True)
cabin_parts.columns = ["Deck", "CabinNum", "Side"]
train_cabin = train.join(cabin_parts)

print("\nBy Cabin Deck:")
print(train_cabin.groupby("Deck")["Transported"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\nBy Cabin Side:")
print(train_cabin.groupby("Side")["Transported"].agg(["mean", "count"]))

Overall Transported rate: 0.504

By CryoSleep:


               mean  count
CryoSleep                 
False      0.328921   5439
True       0.817583   3037

By HomePlanet:
                mean  count
HomePlanet                 
Earth       0.423946   4602
Europa      0.658846   2131
Mars        0.523024   1759

By Cabin Deck:


          mean  count
Deck                 
B     0.734275    779
C     0.680054    747
G     0.516217   2559
A     0.496094    256
F     0.439871   2794
D     0.433054    478
E     0.357306    876
T     0.200000      5

By Cabin Side:
          mean  count
Side                 
P     0.451260   4206
S     0.555037   4288


## Group size vs Transported

We never closed the loop on this one. Test it directly, and also check whether it's just riding on `CryoSleep`/`Cabin` (i.e. confounded) rather than adding independent signal.

In [9]:
train_grp = train.copy()
train_grp["Group"] = train_grp["PassengerId"].str.split("_").str[0]
train_grp["GroupSize"] = train_grp.groupby("Group")["PassengerId"].transform("size")

print("By GroupSize:")
print(train_grp.groupby("GroupSize")["Transported"].agg(["mean", "count"]))

print("\nSolo (GroupSize==1) vs traveling with others (GroupSize>1):")
train_grp["IsSolo"] = train_grp["GroupSize"] == 1
print(train_grp.groupby("IsSolo")["Transported"].agg(["mean", "count"]))

By GroupSize:


               mean  count
GroupSize                 
1          0.452445   4805
2          0.538050   1682
3          0.593137   1020
4          0.640777    412
5          0.592453    265
6          0.614943    174
7          0.541126    231
8          0.394231    104

Solo (GroupSize==1) vs traveling with others (GroupSize>1):
            mean  count
IsSolo                 
False   0.566872   3888
True    0.452445   4805
